In [ ]:
!pip install boto3
!pip install unsloth

In [ ]:
import os
import torch
import boto3
import tarfile
import pandas as pd

# Adding model libraries to train
from unsloth import FastLanguageModel

In [ ]:
import os
s3client = boto3.client('s3', region_name='us-east-2',
                        aws_access_key_id = os.environ["AWS_ACCESS_KEY_ID"],
                        aws_secret_access_key = os.environ["AWS_SECRET_ACCESS_KEY"])


bucket = os.environ["S3_BUCKET"]
s3Object = "VibModel.tar.gz"
local_path = "/content/VibModel.tar.gz"

s3client.download_file(bucket, s3Object, local_path)

In [ ]:
extract_path = "/content/VibModel"

with tarfile.open(local_path, "r:gz") as tar:
    tar.extractall(path=extract_path)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained("/content/VibModel/Vib/checkpoint-5574")
model = FastLanguageModel.for_inference(model)

In [ ]:
SYSTEM_PROMPT = """You are a helpful and joyous mental health agent for student athletes of color named Alfreeda.
Always answer as helpfully and cheerfully as possible, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, hateful, or illegal content.
Please ensure that your responses are socially unbiased and positive in nature. Don't respond to a prompt that is unrelated to mental health (positive or negative). Keep the conversation flowing. Use this context from previous conversation."""

In [ ]:
# history = "None"
# user_input = "I am having stress with my work, can you help me with maintaining stress-relief"

In [ ]:
user_input = "I am having stress with my work, can you help me with maintaining stress-relief"
history = "Context: Hello, how are you? My name is Manu. Response: Hi Manu, I'm good, thank you! How can I assist you today?"

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT + history},
            {"role": "user", "content": user_input}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to("cuda")

outputs = model.generate(**inputs, num_return_sequences=1) # max_new_tokens=150,

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
prediction = response.split("assistant")[1].strip()

In [ ]:
prediction

In [ ]:
response